# R6 - Same R1 tools through the managed MCP server + evidence + traced agent

GenAi-2 (Paper 5) R6: reach the four R1 UC functions through **MCP** instead of hand-wired LangChain wrappers, over one of the Paper 4 probe claims.

Deliverables:
1. `evidence/r6_mcp.json` - keys `transport` (managed; app/stdio are permitted fallbacks), `server` (contains `/api/2.0/mcp/functions/claims/genai`), `tools` (the four R1 tools; the `claims__genai__` prefix is accepted), `calls` (objects with `tool`, `arguments` keyed by the tool's parameter names, e.g. `claim_id`, and `result`, the text the tool returned).
2. MCP calls check - a `lookup_claim` call for one of the Paper 4 probe claims, and a `compute_payable` call for the SAME claim whose result equals the verified answer.
3. `agent/traces/r6_mcp_agent.json` - the same keys as the R2 traces (trace_id, tool_calls, final_answer, cited_chunk_ids): an agent run with tool calls and a trace_id that exists in MLflow.

Design: DatabricksMCPClient (notebook auth) to the managed MCP functions server; a small bridge turns the four MCP tools into LangChain tools with the plain R1 names, so the agent run keeps the exact R2 shape; the verified answer is recomputed from the R1 rules engine and cross-checked against the MCP compute_payable result.

In [0]:
# notebook-scoped install (R5 pattern: NO kernel restart - the running kernel keeps the
# protobuf it booted with). Pinned versions to skip pip resolver backtracking:
#   - langchain-databricks 0.1.2 + langchain-core 0.3.86 (R2-R5 agent pattern)
#   - mlflow-skinny<3 (R5 lesson)
#   - databricks-mcp 0.1.0: NEWER databricks-mcp needs databricks-ai-bridge/databricks-ai-search
#     which requires databricks-vectorsearch>=0.41 - incompatible with langchain-databricks==0.1.2's
#     vectorsearch<0.41 pin, so pip silently falls back to 0.1.0. 0.1.0 has no DatabricksMCPClient,
#     only the OAuth auth provider - fine: we speak MCP directly with the mcp SDK.
#   - mcp==1.16.0: the well-known client API (streamablehttp_client(url, headers=...));
#     mcp 2.x renamed everything (streamable_http_client + httpx2 transports).
%pip install "langchain-databricks==0.1.2" "langchain-core==0.3.86" "mlflow-skinny<3" "databricks-mcp==0.1.0" "mcp==1.16.0"
%pip install "protobuf>=6.31,<7"

In [0]:
# 1. R6 config (same constants as R1-R5) + paths. The MCP server URL is built in the connect cell.
CHAT_MODEL = "databricks-meta-llama-3-3-70b-instruct"
TEMPERATURE = 0.0            # inspectable, repeatable runs
GENAI2_ROOT = "/Workspace/Users/custom001x@gmail.com/saffron-shield/GenAi-2"
EVIDENCE_PATH = GENAI2_ROOT + "/evidence/r6_mcp.json"
AGENT_PATH = GENAI2_ROOT + "/agent/traces/r6_mcp_agent.json"

import mlflow
mlflow.set_tracking_uri("databricks")
mlflow.set_experiment("/Users/custom001x@gmail.com/saffron-shield/genai2-agent")
EXP = mlflow.get_experiment_by_name("/Users/custom001x@gmail.com/saffron-shield/genai2-agent")
print(f"CHAT_MODEL={CHAT_MODEL} | temperature={TEMPERATURE}")
print(f"MLflow experiment id={EXP.experiment_id}")

In [0]:
# 2. Ground truth: pick one claim from the Paper 4 probe list, fetch its facts, recompute the
#    verified answer with the R1 rules engine - this is what the MCP compute_payable result must equal.
import json

PROBE_CSV = "/Volumes/claims/raw/landing/genai/adjudication_probe_claims.csv"
pdf = spark.read.option("header", True).csv(PROBE_CSV)
col = next((c for c in pdf.columns if "claim" in c.lower()), pdf.columns[0])
ids = [str(r[0]) for r in pdf.select(col).collect() if r[0] and str(r[0]).upper().startswith("CLM")]
assert ids, f"no claim ids found in {PROBE_CSV}"
PROBE_CLAIM_ID = ids[0]
print(f"probe claims available: {len(ids)} | using {PROBE_CLAIM_ID} (first of the Paper 4 probe list)")

facts = spark.sql(f"SELECT * FROM claims.genai.lookup_claim('{PROBE_CLAIM_ID}')").first().asDict()
print("claim facts:", {k: facts[k] for k in ("plan_code_at_admission", "room_rent_per_day", "room_rent_cap_per_day",
                                               "is_network_at_admission", "is_pre_existing", "policy_tenure_months",
                                               "days_since_policy_start", "billed_amount")})

vs = spark.sql(f"SELECT claims.genai.compute_payable('{facts['plan_code_at_admission']}', "
               f"{str(bool(facts['is_pre_existing'])).lower()}, {int(facts['policy_tenure_months'])}, "
               f"{int(facts['days_since_policy_start'])}, {str(bool(facts['is_network_at_admission'])).lower()}, "
               f"{int(facts['room_rent_per_day'])}, CAST({float(facts['billed_amount'])} AS DOUBLE)) AS s").first().s
EXPECTED_VERDICT = json.loads(vs)
print("verified answer (R1 rules engine):", EXPECTED_VERDICT)

PROBE_QUESTION = f"What is the status of claim {PROBE_CLAIM_ID}?"
print("agent probe question:", PROBE_QUESTION)

In [0]:
# 3. Connect to the MCP server for the claims.genai UC functions and confirm all four R1 tools
#    are exposed (the claims__genai__ prefix is accepted). Preferred transport: Databricks'
#    MANAGED MCP server (notebook auth). If it is not enabled in this workspace, use the
#    PERMITTED FALLBACK: a local stdio MCP server exposing the same four R1 UC functions,
#    executed through the SQL Statement Execution API on the workspace SQL warehouse.
import asyncio
import json
import os
import sys

# pip upgraded mcp 2.3.0 -> 1.16.0 mid-session; drop stale modules from sys.modules so the
# imports below load the 1.16.0 files from disk (NO kernel restart - R5 lesson).
for _m in [m for m in list(sys.modules)
           if m == "mcp" or m.startswith("mcp.") or m.startswith("mcp_types")
           or m.startswith("httpx2") or m.startswith("httpcore2")]:
    del sys.modules[_m]

from databricks.sdk import WorkspaceClient
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client
from mcp.client.streamable_http import streamablehttp_client

HOST = spark.conf.get("spark.databricks.workspaceUrl")
MCP_SERVER_URL = f"https://{HOST}/api/2.0/mcp/functions/claims/genai"
AUTH_HEADERS = WorkspaceClient().config.authenticate()
assert AUTH_HEADERS.get("Authorization", "").startswith("Bearer "), AUTH_HEADERS.keys()
print("managed MCP server:", MCP_SERVER_URL)

def run_async(coro):
    """Run a coroutine even when the notebook kernel already has an event loop running:
    direct asyncio.run when idle, otherwise a fresh loop on a worker thread."""
    try:
        asyncio.get_running_loop()
    except RuntimeError:
        return asyncio.run(coro)
    import concurrent.futures
    with concurrent.futures.ThreadPoolExecutor(max_workers=1) as ex:
        return ex.submit(asyncio.run, coro).result()

def _exc_tree_msgs(exc) -> str:
    subs = getattr(exc, "exceptions", None)
    if not subs:
        return str(exc)
    return " | ".join(_exc_tree_msgs(s) for s in subs)

# ---- attempt 1: managed transport -----------------------------------------
async def _managed_list_tools() -> list:
    async with streamablehttp_client(MCP_SERVER_URL, headers=AUTH_HEADERS) as (read, write, _sid):
        async with ClientSession(read, write) as session:
            await session.initialize()
            resp = await session.list_tools()
            return [(t.name, t.description or "") for t in resp.tools]

MANAGED_ERROR = None
try:
    MCP_NAMES = [n for n, _d in run_async(_managed_list_tools())]
    TRANSPORT = "managed"
    SERVER_DESC = MCP_SERVER_URL
    print("transport = MANAGED")
except BaseException as e:
    MANAGED_ERROR = _exc_tree_msgs(e)
    assert ("not enabled" in MANAGED_ERROR.lower() or "not_found" in MANAGED_ERROR.lower()), \
        f"managed MCP server failed for an unexpected reason: {MANAGED_ERROR}"
    print(f"managed MCP servers NOT enabled in this workspace - using the permitted stdio fallback")
    TRANSPORT = "stdio"

if TRANSPORT == "stdio":
    # ---- fallback: local MCP server over stdio ----------------------------
    WAREHOUSE_ID = "78d419cb76b1d7e9"   # Serverless Starter Warehouse (workspace SQL warehouse)
    SERVER_SCRIPT = GENAI2_ROOT + "/agent/mcp_stdio_server.py"

    SERVER_SOURCE = '''
# MCP stdio server exposing the four Saffron Shield R1 UC functions as MCP tools.
# Permitted-fallback transport: Databricks managed MCP servers (Public Preview) are not
# enabled in this workspace, so this local server speaks MCP over stdio and executes the
# UC functions through the SQL Statement Execution API on the workspace SQL warehouse.
import json
import os
import time

from databricks.sdk import WorkspaceClient
from databricks.sdk.service.sql import Disposition, Format, StatementState
from mcp.server.fastmcp import FastMCP

WAREHOUSE_ID = os.environ["MCP_WAREHOUSE_ID"]
w = WorkspaceClient()

def q(s) -> str:
    return str(s).replace(chr(39), chr(39) + chr(39))

def run_sql(sql: str) -> list:
    resp = w.statement_execution.execute_statement(
        warehouse_id=WAREHOUSE_ID, statement=sql,
        disposition=Disposition.INLINE, format=Format.JSON_ARRAY, wait_timeout="50s")
    while resp.status.state in (StatementState.PENDING, StatementState.RUNNING):
        time.sleep(2)
        resp = w.statement_execution.get_statement(resp.statement_id)
    if resp.status.error is not None:
        raise RuntimeError(f"statement failed: {resp.status.error.error_code} {resp.status.error.message}")
    if resp.status.state != StatementState.SUCCEEDED:
        raise RuntimeError(f"statement ended in state {resp.status.state}")
    cols = [c.name for c in resp.manifest.schema.columns]
    rows = [dict(zip(cols, r)) for r in (resp.result.data_array or [])]
    result = resp.result
    while result.next_chunk_index is not None:
        result = w.statement_execution.get_statement_result_chunk_n(resp.statement_id, result.next_chunk_index)
        rows.extend(dict(zip(cols, r)) for r in (result.data_array or []))
    return rows

mcp = FastMCP("claims-genai")

@mcp.tool()
def lookup_claim(claim_id: str) -> str:
    "Look up ONE insurance claim by its exact claim id (example: CLM-10642). Returns every stored fact of that claim as JSON: member_id, policy_id, hospital_id, status, claim_type, admission and discharge dates, billed and approved amounts, plan code at admission, room rent and cap, network flag, policy start date and tenure, days since policy start. Use for ANY question about a specific claim."
    return json.dumps(run_sql(f"SELECT * FROM claims.genai.lookup_claim('{q(claim_id)}')"), default=str)

@mcp.tool()
def plan_at(member_id: str, as_of_date: str) -> str:
    "Look up the Saffron Shield plan terms that were VALID on a given date (YYYY-MM-DD) for a member id (example: MEM-0493). Returns plan_code, plan_family, room_rent_cap_per_day, copay_pct, ped_waiting_months, initial_waiting_days, valid_from, valid_to. Use with the member id and admission date from lookup_claim when a claim question needs plan terms."
    return json.dumps(run_sql(f"SELECT * FROM claims.genai.plan_at('{q(member_id)}', DATE'{q(as_of_date)}')"), default=str)

@mcp.tool()
def search_policy(query: str) -> str:
    "Search the Saffron Shield policy binder (hybrid keyword + semantic). Returns up to 5 chunks with chunk_id, doc_name, heading and full text. Use for ANY question about what the policy says: waiting periods, room rent caps, co-payments, network hospitals, exclusions. Always read each chunk's 'Applies to' line before quoting it."
    return json.dumps(run_sql(f"SELECT * FROM claims.genai.search_policy('{q(query)}')"), default=str)

@mcp.tool()
def compute_payable(plan_code: str, is_pre_existing: bool, policy_tenure_months: int, days_since_policy_start: int, is_network_at_admission: bool, room_rent_per_day: int, billed_amount: float) -> str:
    "Decide how much a claim pays, strictly by the policy Definitions. Pass the claim facts from lookup_claim (tenure months, days since policy start, pre-existing flag, network flag, room rent per day, billed amount) and the plan_code from plan_at. Returns JSON with payable, reason, room_rent_deduction_pct, copay_pct and payable_amount. Never decide a payout without this tool."
    pe = "true" if is_pre_existing else "false"
    nw = "true" if is_network_at_admission else "false"
    rows = run_sql(f"SELECT claims.genai.compute_payable('{q(plan_code)}', {pe}, {int(policy_tenure_months)}, {int(days_since_policy_start)}, {nw}, {int(room_rent_per_day)}, CAST({float(billed_amount)} AS DOUBLE)) AS s")
    return rows[0]["s"]

if __name__ == "__main__":
    mcp.run()
'''
    dbutils.fs.put(SERVER_SCRIPT, SERVER_SOURCE, overwrite=True)
    print("stdio MCP server written:", SERVER_SCRIPT)

    CHILD_ENV = dict(os.environ)
    CHILD_ENV["DATABRICKS_HOST"] = f"https://{HOST}"
    CHILD_ENV["DATABRICKS_TOKEN"] = AUTH_HEADERS["Authorization"][len("Bearer "):]
    CHILD_ENV["MCP_WAREHOUSE_ID"] = WAREHOUSE_ID
    SERVER_PARAMS = StdioServerParameters(command=sys.executable, args=[SERVER_SCRIPT], env=CHILD_ENV)

    async def _stdio_list_tools() -> list:
        async with stdio_client(SERVER_PARAMS) as (read, write):
            async with ClientSession(read, write) as session:
                await session.initialize()
                resp = await session.list_tools()
                return [(t.name, t.description or "") for t in resp.tools]

    async def _stdio_call_tool(name: str, arguments: dict) -> str:
        async with stdio_client(SERVER_PARAMS) as (read, write):
            async with ClientSession(read, write) as session:
                await session.initialize()
                res = await session.call_tool(name, arguments)
                return chr(10).join(getattr(c, "text", "") for c in res.content if getattr(c, "text", None))

    MCP_NAMES = [n for n, _d in run_async(_stdio_list_tools())]
    TRANSPORT = "stdio"
    SERVER_DESC = (f"stdio: python {SERVER_SCRIPT} on SQL warehouse {WAREHOUSE_ID} "
                   f"(managed {MCP_SERVER_URL} not enabled in this workspace: Public Preview off)")
    print("transport = STDIO (permitted fallback)")

print(f"MCP server exposes {len(MCP_NAMES)} tool(s):", MCP_NAMES)

BASE_TOOLS = ["lookup_claim", "plan_at", "search_policy", "compute_payable"]

def mcp_name(base: str) -> str:
    """The MCP tool name for an R1 tool: plain, or with the claims__genai__ prefix."""
    cands = [n for n in MCP_NAMES if n == base or n.endswith("__" + base)]
    assert cands, f"{base} not exposed by the MCP server: {MCP_NAMES}"
    return cands[0]

for b in BASE_TOOLS:
    mcp_name(b)
print("all four R1 tools exposed by the MCP server")

def call_mcp(base: str, arguments: dict) -> tuple:
    """Call one R1 tool through MCP; returns (mcp tool name, result text)."""
    if TRANSPORT == "managed":
        async def _call(name, arguments):
            async with streamablehttp_client(MCP_SERVER_URL, headers=AUTH_HEADERS) as (read, write, _sid):
                async with ClientSession(read, write) as session:
                    await session.initialize()
                    res = await session.call_tool(name, arguments)
                    return chr(10).join(getattr(c, "text", "") for c in res.content if getattr(c, "text", None))
    else:
        _call = _stdio_call_tool
    name = mcp_name(base)
    return name, run_async(_call(name, arguments))

In [0]:
# 4. The two required MCP calls: lookup_claim for the probe claim, then compute_payable for the
#    SAME claim; the compute result must equal the verified answer. Each call is recorded as
#    {tool, arguments (keyed by the tool's parameter names), result (the text the tool returned)}.
CALLS = []

lk_name, lk_text = call_mcp("lookup_claim", {"claim_id": PROBE_CLAIM_ID})
assert PROBE_CLAIM_ID in lk_text, f"lookup_claim result does not mention {PROBE_CLAIM_ID}: {lk_text[:200]}"
CALLS.append({"tool": lk_name, "arguments": {"claim_id": PROBE_CLAIM_ID}, "result": lk_text})
print(f"lookup_claim OK via MCP ({lk_name}): {len(lk_text)} chars of claim facts")

cp_args = {"plan_code": facts["plan_code_at_admission"],
           "is_pre_existing": bool(facts["is_pre_existing"]),
           "policy_tenure_months": int(facts["policy_tenure_months"]),
           "days_since_policy_start": int(facts["days_since_policy_start"]),
           "is_network_at_admission": bool(facts["is_network_at_admission"]),
           "room_rent_per_day": int(facts["room_rent_per_day"]),
           "billed_amount": float(facts["billed_amount"])}
cp_name, cp_text = call_mcp("compute_payable", cp_args)

def verdict_from_text(text: str):
    """The verdict JSON from a tool result text (tolerates surrounding quoting)."""
    for cand in (text, text.strip().strip('"')):
        try:
            j = json.loads(cand)
        except Exception:
            continue
        if isinstance(j, dict) and "payable_amount" in j:
            return j
    return None

def _close(a, b) -> bool:
    return abs(float(a) - float(b)) < 0.005

NUMERIC = ("room_rent_deduction_pct", "payable_amount")

got = verdict_from_text(cp_text)
assert got is not None, f"compute_payable MCP result is not a verdict JSON: {cp_text[:300]}"
equal = all((_close(got[k], EXPECTED_VERDICT[k]) if k in NUMERIC else got[k] == EXPECTED_VERDICT[k])
            for k in EXPECTED_VERDICT)
assert equal, f"MCP compute_payable != verified answer: {got} vs {EXPECTED_VERDICT}"
CALLS.append({"tool": cp_name, "arguments": cp_args, "result": cp_text})
print(f"compute_payable OK via MCP ({cp_name}): result EQUALS the verified answer "
      f"(payable {EXPECTED_VERDICT['payable_amount']}, reason {EXPECTED_VERDICT['reason']})")

In [0]:
# 5. Agent over MCP tools: the exact R2 pattern (ChatDatabricks + manual loop + mlflow trace,
#    system_v1.md prompt), but every tool call goes through the managed MCP client via a small
#    bridge. Tool names stay the plain R1 names so the trace has the same shape as R2's.
try:
    mlflow.langchain.autolog()
except Exception as e:
    print(f"langchain autolog unavailable ({e}); TOOL/CHAIN spans still trace every run")

import json
from mlflow.entities import SpanType
from langchain_databricks import ChatDatabricks
from langchain_core.tools import tool
from langchain_core.messages import SystemMessage, HumanMessage, ToolMessage

@tool
def lookup_claim(claim_id: str) -> str:
    "Look up ONE insurance claim by its exact claim id (example: CLM-10642). Returns every stored fact of that claim as JSON: member_id, policy_id, hospital_id, status, claim_type, admission and discharge dates, billed and approved amounts, plan code at admission, room rent and cap, network flag, policy start date and tenure, days since policy start. Use for ANY question about a specific claim."
    return call_mcp("lookup_claim", {"claim_id": claim_id})[1]

@tool
def plan_at(member_id: str, as_of_date: str) -> str:
    "Look up the Saffron Shield plan terms that were VALID on a given date (YYYY-MM-DD) for a member id (example: MEM-0493). Returns plan_code, plan_family, room_rent_cap_per_day, copay_pct, ped_waiting_months, initial_waiting_days, valid_from, valid_to. Use with the member id and admission date from lookup_claim when a claim question needs plan terms."
    return call_mcp("plan_at", {"member_id": member_id, "as_of_date": as_of_date})[1]

@tool
def search_policy(query: str) -> str:
    "Search the Saffron Shield policy binder (hybrid keyword + semantic). Returns up to 5 chunks with chunk_id, doc_name, heading and full text. Use for ANY question about what the policy says: waiting periods, room rent caps, co-payments, network hospitals, exclusions. Always read each chunk's 'Applies to' line before quoting it."
    return call_mcp("search_policy", {"query": query})[1]

@tool
def compute_payable(plan_code: str, is_pre_existing: bool, policy_tenure_months: int,
                    days_since_policy_start: int, is_network_at_admission: bool,
                    room_rent_per_day: int, billed_amount: float) -> str:
    "Decide how much a claim pays, strictly by the policy Definitions. Pass the claim facts from lookup_claim (tenure months, days since policy start, pre-existing flag, network flag, room rent per day, billed amount) and the plan_code from plan_at. Returns JSON with payable, reason, room_rent_deduction_pct, copay_pct and payable_amount. Never decide a payout without this tool."
    return call_mcp("compute_payable", {"plan_code": plan_code, "is_pre_existing": is_pre_existing,
                                        "policy_tenure_months": policy_tenure_months,
                                        "days_since_policy_start": days_since_policy_start,
                                        "is_network_at_admission": is_network_at_admission,
                                        "room_rent_per_day": room_rent_per_day,
                                        "billed_amount": billed_amount})[1]

TOOL_FNS = {t.name: t for t in [lookup_claim, plan_at, search_policy, compute_payable]}
assert set(TOOL_FNS) == set(BASE_TOOLS)
SYSTEM_PROMPT = dbutils.fs.head(GENAI2_ROOT + "/agent/prompts/system_v1.md", 100000)

@mlflow.trace(span_type=SpanType.CHAIN)
def run_agent(question: str) -> dict:
    """One agent run over MCP-bridged tools; local call list (no shared state)."""
    calls = []
    llm = ChatDatabricks(endpoint=CHAT_MODEL, temperature=TEMPERATURE).bind_tools(list(TOOL_FNS.values()))
    messages = [SystemMessage(content=SYSTEM_PROMPT), HumanMessage(content=question)]
    final = None
    for _step in range(5):
        ai = llm.invoke(messages)
        messages.append(ai)
        if not ai.tool_calls:
            final = ai.content
            break
        for tc in ai.tool_calls:
            out = TOOL_FNS[tc["name"]].invoke(tc["args"])
            calls.append({"name": tc["name"], "args": tc["args"], "output": str(out)})
            messages.append(ToolMessage(content=str(out), tool_call_id=tc["id"]))
    assert final is not None, "agent produced no final answer"
    cited = []
    for c in calls:
        if c["name"] == "search_policy":
            try:
                cited.extend([r["chunk_id"] for r in json.loads(c["output"])])
            except Exception:
                pass
    return {"answer": final, "tool_calls": [c["name"] for c in calls], "cited_chunk_ids": cited}

res = run_agent(PROBE_QUESTION)
tid = mlflow.get_last_active_trace_id()
assert tid, "no MLflow trace after the agent run"
agent_payload = {"trace_id": tid, "tool_calls": res["tool_calls"],
                 "final_answer": res["answer"], "cited_chunk_ids": res["cited_chunk_ids"]}
dbutils.fs.put(AGENT_PATH, json.dumps(agent_payload, indent=2), overwrite=True)
print("agent over MCP:", res["tool_calls"])
print("final answer:", res["answer"][:300])
print("trace:", tid)
print("agent trace written:", AGENT_PATH)

In [0]:
# 7. Verify every R6 rule from the PERSISTED files (fresh recomputation of the verified answer).
import json
import mlflow

ev = json.loads(dbutils.fs.head(EVIDENCE_PATH, 400000))
assert list(ev.keys()) == ["transport", "server", "tools", "calls"], list(ev.keys())
assert ev["transport"] in ("managed", "app", "stdio"), ev["transport"]
if ev["transport"] == "managed":
    assert "/api/2.0/mcp/functions/claims/genai" in ev["server"], ev["server"]
else:
    assert ev["transport"] == "stdio" and "stdio" in ev["server"] and "mcp_stdio_server.py" in ev["server"], ev["server"]
    print("(managed MCP servers are not enabled in this workspace - stdio is the permitted fallback)")
for b in BASE_TOOLS:
    assert any(n == b or n.endswith("__" + b) for n in ev["tools"]), (b, ev["tools"])
print(f"evidence OK: transport={ev['transport']} | server recorded | all 4 R1 tools listed")

def loose(name: str, base: str) -> bool:
    return name == base or name.endswith("__" + base)

lk = next(c for c in ev["calls"] if loose(c["tool"], "lookup_claim"))
assert lk["arguments"].get("claim_id") == PROBE_CLAIM_ID and PROBE_CLAIM_ID in lk["result"], \
    (lk["arguments"], lk["result"][:200])
cp = next(c for c in ev["calls"] if loose(c["tool"], "compute_payable"))
assert set(cp["arguments"]) == {"plan_code", "is_pre_existing", "policy_tenure_months",
                                "days_since_policy_start", "is_network_at_admission",
                                "room_rent_per_day", "billed_amount"}, cp["arguments"]

f2 = spark.sql(f"SELECT * FROM claims.genai.lookup_claim('{PROBE_CLAIM_ID}')").first().asDict()
s2 = spark.sql(f"SELECT claims.genai.compute_payable('{f2['plan_code_at_admission']}', "
               f"{str(bool(f2['is_pre_existing'])).lower()}, {int(f2['policy_tenure_months'])}, "
               f"{int(f2['days_since_policy_start'])}, {str(bool(f2['is_network_at_admission'])).lower()}, "
               f"{int(f2['room_rent_per_day'])}, CAST({float(f2['billed_amount'])} AS DOUBLE)) AS s").first().s
exp2 = json.loads(s2)
got2 = verdict_from_text(cp["result"])
assert got2 is not None, "persisted compute_payable result does not parse as a verdict"
equal2 = all((_close(got2[k], exp2[k]) if k in NUMERIC else got2[k] == exp2[k]) for k in exp2)
assert equal2, (got2, exp2)
print(f"MCP calls OK: lookup_claim({PROBE_CLAIM_ID}) returned the claim facts | compute_payable result EQUALS "
      f"the freshly recomputed verified answer (payable {exp2['payable_amount']}, {exp2['reason']})")

tr = json.loads(dbutils.fs.head(AGENT_PATH, 200000))
assert list(tr.keys()) == ["trace_id", "tool_calls", "final_answer", "cited_chunk_ids"], list(tr.keys())
assert mlflow.get_trace(tr["trace_id"]) is not None, "agent trace_id not found in MLflow"
assert tr["tool_calls"] and all(isinstance(t, str) for t in tr["tool_calls"]), tr["tool_calls"]
assert tr["final_answer"].strip(), "final_answer is empty"
assert isinstance(tr["cited_chunk_ids"], list), type(tr["cited_chunk_ids"])
print(f"agent trace OK: same keys as the R2 traces | trace_id in MLflow | tool_calls={tr['tool_calls']}")
print("ALL R6 SELF-CHECKS PASSED")

In [0]:
# 6. Write evidence/r6_mcp.json - transport, server, tools, calls (exact key order).
import json

payload = {"transport": TRANSPORT, "server": SERVER_DESC, "tools": MCP_NAMES, "calls": CALLS}
dbutils.fs.put(EVIDENCE_PATH, json.dumps(payload, indent=2), overwrite=True)
print("evidence written:", EVIDENCE_PATH)
print(f"transport={TRANSPORT} | {len(payload['tools'])} tool(s) | {len(payload['calls'])} call(s)")